# 5 – Selezione del modello spaziale (Province italiane)

**SPEC 2026** – Econometria Spaziale (Python Lab)

Equivalente Python di `R/5_spatial_model_selection.R`

- NEET ~ Formazione continua
- LISA su NEET
- OLS + LM tests
- SAR (spatial lag), SEM (spatial error), SARAR

In [ ]:
!pip install -q geopandas pyarrow libpysal esda spreg mapclassify

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from shapely.geometry import LineString
from libpysal.weights import Queen, lag_spatial
from esda import Moran, Moran_Local
from spreg import OLS, ML_Lag, ML_Error, GM_Combo
import warnings
warnings.filterwarnings('ignore')
np.random.seed(123)

BASE = "https://github.com/vincnardelli/spec/raw/main/data"
for f in ["tanzania.parquet", "kc_house.parquet", "kc_grid.parquet",
          "italian_provinces.parquet", "visium_hne_points.parquet"]:
    if not os.path.exists(f):
        !wget -q {BASE}/{f}

## 1) Caricamento e preparazione dati

In [ ]:
data = gpd.read_parquet("italian_provinces.parquet")
data = data[["den_prov", "giovani_non_lavorano_non_studiano",
             "partecipazione_formazione_continua", "geometry"]].copy()
data.columns = ["den_prov", "neet", "formazione", "geometry"]
data.head()

## 2) Analisi esplorativa

In [ ]:
print(f"Correlazione: {data['formazione'].corr(data['neet']):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(data["formazione"], data["neet"], alpha=0.7, edgecolors="k", lw=0.3)
axes[0].set_xlabel("Formazione continua")
axes[0].set_ylabel("NEET")
axes[0].set_title("Scatter")

data.plot(column="neet", legend=True, cmap="Reds",
          legend_kwds={"label": "NEET (%)"}, ax=axes[1])
axes[1].set_title("NEET per provincia")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

## 3) Pesi spaziali e LISA

In [ ]:
w = Queen.from_dataframe(data, silence_warnings=True)
w.transform = "r"
print(w)

In [ ]:
centroids = data.geometry.centroid
lines = []
for i in w.neighbors:
    for j in w.neighbors[i]:
        lines.append(LineString([centroids.iloc[i], centroids.iloc[j]]))
network = gpd.GeoDataFrame(geometry=lines, crs=data.crs)

fig, ax = plt.subplots(figsize=(6, 8))
data.plot(ax=ax, facecolor="white", edgecolor="grey")
network.plot(ax=ax, color="#404040", linewidth=0.3)
ax.set_title("Rete di contiguit\u00e0 Queen")
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
y_neet = data["neet"].values
mi_neet = Moran(y_neet, w, permutations=999)
print(f"Moran's I (NEET): {mi_neet.I:.4f}, p-value: {mi_neet.p_sim:.4f}")

In [ ]:
lisa = Moran_Local(y_neet, w, permutations=999)

neet_mean = y_neet.mean()
neet_lag = lag_spatial(w, y_neet)
lag_mean = neet_lag.mean()

cluster = pd.Series("NS", index=data.index)
sig = lisa.p_sim < 0.05
cluster[(sig) & (y_neet > neet_mean) & (neet_lag > lag_mean)] = "HH"
cluster[(sig) & (y_neet < neet_mean) & (neet_lag < lag_mean)] = "LL"
cluster[(sig) & (y_neet > neet_mean) & (neet_lag < lag_mean)] = "HL"
cluster[(sig) & (y_neet < neet_mean) & (neet_lag > lag_mean)] = "LH"
data["cluster"] = cluster

lisa_colors = {"HH": "#ca0020", "LL": "#0571b0",
               "HL": "#f4a582", "LH": "#92c5de", "NS": "lightgray"}

fig, ax = plt.subplots(figsize=(6, 8))
data.plot(color=data["cluster"].map(lisa_colors), edgecolor="white",
          linewidth=0.2, ax=ax)
handles = [plt.Rectangle((0,0),1,1, facecolor=c) for c in lisa_colors.values()]
ax.legend(handles, lisa_colors.keys(), title="LISA", loc="lower left")
ax.set_title("LISA \u2013 NEET")
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 4) OLS e diagnostica spaziale

Il summary di `spreg.OLS` con `spat_diag=True` riporta automaticamente i test LM.

In [ ]:
y = data[["neet"]].values
x = data[["formazione"]].values

ols = OLS(y, x, w=w, spat_diag=True,
          name_y="neet", name_x=["formazione"])
print(ols.summary)

In [ ]:
data["residuals"] = ols.u.flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(data["formazione"], data["neet"], alpha=0.7,
                edgecolors="k", lw=0.3)
b0, b1 = float(ols.betas[0]), float(ols.betas[1])
xr = np.linspace(data["formazione"].min(), data["formazione"].max())
axes[0].plot(xr, b0 + b1 * xr, "r-")
axes[0].set_xlabel("Formazione")
axes[0].set_ylabel("NEET")
axes[0].set_title("OLS")

norm = TwoSlopeNorm(vmin=data["residuals"].min(), vcenter=0,
                    vmax=data["residuals"].max())
data.plot(column="residuals", legend=True, cmap="RdBu_r", norm=norm,
          ax=axes[1])
axes[1].set_title("Mappa residui OLS")
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
mi_resid = Moran(ols.u.flatten(), w, permutations=999)
print(f"Moran's I (residui OLS): {mi_resid.I:.4f}, p-value: {mi_resid.p_sim:.4f}")

## 5) Spatial Lag Model (SAR)

$y = \rho W y + X \beta + \varepsilon$

In [ ]:
sar = ML_Lag(y, x, w=w,
             name_y="neet", name_x=["formazione"])
print(sar.summary)

In [ ]:
rho = float(sar.betas[-1])
beta_k = float(sar.betas[1])
W_dense = w.full()[0]
n = W_dense.shape[0]
I_n = np.eye(n)
S_k = np.linalg.inv(I_n - rho * W_dense) * beta_k

direct = np.mean(np.diag(S_k))
total = np.mean(S_k.sum(axis=0))
indirect = total - direct

print("=== Impatti spaziali (SAR) ===")
print(f"Direct:   {direct:.3f}")
print(f"Indirect: {indirect:.3f}")
print(f"Total:    {total:.3f}")

## 6) Spatial Error Model (SEM)

$y = X\beta + u, \quad u = \lambda W u + \varepsilon$

In [ ]:
sem = ML_Error(y, x, w=w,
               name_y="neet", name_x=["formazione"])
print(sem.summary)

## 7) SARAR Model

$y = \rho W y + X\beta + u, \quad u = \lambda W u + \varepsilon$

In [ ]:
sarar = GM_Combo(y, x, w=w,
                 name_y="neet", name_x=["formazione"])
print(sarar.summary)

In [ ]:
rho_sarar = float(sarar.betas[-1])
beta_k_sarar = float(sarar.betas[1])
S_sarar = np.linalg.inv(I_n - rho_sarar * W_dense) * beta_k_sarar

direct_s = np.mean(np.diag(S_sarar))
total_s = np.mean(S_sarar.sum(axis=0))
indirect_s = total_s - direct_s

print("=== Impatti spaziali (SARAR) ===")
print(f"Direct:   {direct_s:.3f}")
print(f"Indirect: {indirect_s:.3f}")
print(f"Total:    {total_s:.3f}")

## 8) Confronto modelli

In [ ]:
print(f"{'Modello':10s} {'Intercept':>10s} {'Formaz.':>10s} {'rho':>10s} {'lambda':>10s} {'AIC':>10s}")
print("-" * 62)
print(f"{'OLS':10s} {float(ols.betas[0]):10.2f} {float(ols.betas[1]):10.2f} {'\u2014':>10s} {'\u2014':>10s} {ols.aic:10.2f}")
print(f"{'SAR':10s} {float(sar.betas[0]):10.2f} {float(sar.betas[1]):10.2f} {float(sar.betas[-1]):10.3f} {'\u2014':>10s} {sar.aic:10.2f}")
print(f"{'SEM':10s} {float(sem.betas[0]):10.2f} {float(sem.betas[1]):10.2f} {'\u2014':>10s} {float(sem.lam):10.3f} {sem.aic:10.2f}")

print(f"\n{'Modello':10s} {'Direct':>10s} {'Indirect':>10s} {'Total':>10s}")
print("-" * 42)
print(f"{'SAR':10s} {direct:10.3f} {indirect:10.3f} {total:10.3f}")
print(f"{'SARAR':10s} {direct_s:10.3f} {indirect_s:10.3f} {total_s:10.3f}")